# GPU 硬件与多卡互联

> 前面几节讨论 DDP、ZeRO、FSDP 时，把 GPU 当作一个抽象的「计算单元」——给它数据，它就返回梯度。但实际上 GPU 内部有清晰的内存层级，多卡之间也有带宽差异巨大的几种连接方式。把这些硬件细节抽象掉，能讲清楚算法；一旦落到真实训练和推理，硬件拓扑往往决定了方案是否可行。
>
> 这一节从单张 GPU 内部开始：HBM 与 SRAM 的容量和带宽差距为什么让 Flash Attention 成为必需。再扩展到多卡互联，对比 PCIe、NVLink、NVSwitch、InfiniBand 的带宽量级，看清 DGX H100 的拓扑结构。最后讨论拓扑感知调度，以及 MoE 的 Expert Parallelism 为什么对网络拓扑特别敏感。


GPU 不是一块等价的「大显存 + 快算力」。一张 H100 内部，靠近计算单元的 SRAM 只有约 20 MB，但带宽高达数十 TB/s；远离计算单元的 HBM 有 80 GB，但带宽只有约 3 TB/s。两者相差 30 倍以上。这个差距决定了哪些计算能跑得快——把数据尽量留在 SRAM 里，避免反复读写 HBM，是高性能 CUDA kernel 设计的核心思路。

多卡层面，PCIe 是通用总线但带宽有限，NVLink 是 GPU 之间的专用高速通道，InfiniBand 是跨节点的网络协议。同样是「两张 GPU 之间传 1 GB 数据」，走 NVLink 和走 PCIe 速度能差 6 到 10 倍。分布式训练中，通信时间往往占总时间的相当比例，所以了解多卡拓扑不是硬件工程师的专利，而是任何想跑大模型的人必须掌握的基础。

本节按「单卡内存层级 → GPU 内部结构 → 多卡互联 → 典型拓扑 → 拓扑感知调度」的顺序展开，最后用 `nvidia-smi` 等工具演示如何在实际机器上查看这些信息。


## 1. 单卡内存层级：HBM 与 SRAM

一张 GPU 内部主要有两段存储。HBM（High Bandwidth Memory）是显存，容量大但相对慢；SRAM（Static Random Access Memory）是片上缓存，容量小但极快。在 NVIDIA 架构里，SRAM 主要指 L1 cache 和 shared memory，统称「片上内存」。

下表列出现代几张主流数据中心 GPU 的内存配置：


In [ ]:
# === 主流数据中心 GPU 的内存配置 ===
# 数据来源：NVIDIA 官方规格表

gpus = [
    # (型号, HBM 容量 GB, HBM 带宽 TB/s, SRAM 容量 MB, 发布年份)
    ("A100 40GB",  40, 1.55,  20, 2020),
    ("A100 80GB",  80, 2.00,  20, 2021),
    ("H100 SXM",   80, 3.35,  20, 2022),
    ("H200 SXM",  141, 4.80,  20, 2024),
    ("B200",      192, 8.00,  40, 2024),
]

print(f"{'型号':<14} {'HBM(GB)':>9} {'HBM带宽(TB/s)':>15} {'SRAM(MB)':>10} {'年份':>6}")
print("-" * 60)
for name, hbm_gb, hbm_bw, sram_mb, year in gpus:
    print(f"{name:<14} {hbm_gb:>9} {hbm_bw:>15.2f} {sram_mb:>10} {year:>6}")

print()
print("关键观察：从 A100 到 B200，HBM 容量涨了 5 倍，带宽涨了约 5 倍，但 SRAM 涨得很慢。")
print("HBM 越大能装下越大的 KV cache 和激活值；带宽越高，读写大 tensor 越快。")


### HBM 与 SRAM 的带宽差距

HBM 的带宽看起来很高（H100 是 3.35 TB/s），但 SRAM 更高。H100 的 shared memory 加 L1 cache 合起来的可用带宽估算在 30 TB/s 量级，比 HBM 快接近 10 倍。这个差距直接影响 attention 的实现方式。


In [ ]:
# === HBM vs SRAM 带宽对比（以 H100 为例） ===
hbm_bw_tb = 3.35       # H100 HBM 带宽，单位 TB/s
sram_bw_tb = 30.0      # H100 SRAM 估算带宽，单位 TB/s（约值）

ratio = sram_bw_tb / hbm_bw_tb
print(f"H100 HBM 带宽:  {hbm_bw_tb:.2f} TB/s")
print(f"H100 SRAM 带宽: {sram_bw_tb:.2f} TB/s（估算）")
print(f"带宽差距: SRAM 是 HBM 的 {ratio:.1f} 倍")
print()
print("如果一个 attention kernel 需要反复读写 N×N 的 attention 矩阵，")
print("每次都经过 HBM，会比把矩阵分块留在 SRAM 内慢很多。")
print("这正是 Flash Attention 优化的出发点：分块计算，让中间结果留在 SRAM。")


### 为什么 Flash Attention 必需

标准的 attention 公式是 $\text{softmax}(QK^T / \sqrt{d}) V$。朴素的实现会先把 $N \times N$ 的 attention 分数矩阵写到 HBM，再读回来做 softmax。序列长度 $N$ 增大时，这个 $N^2$ 矩阵的显存和读写都爆炸。

Flash Attention 的核心是把 Q、K、V 切成块，每块的中间结果小到能放进 SRAM， softmax 用 online 算法（边累加边归一化）逐块算完，不必把完整 $N^2$ 矩阵落到 HBM。这样 attention 的 HBM 读写从 $O(N^2)$ 降到 $O(N)$，激活显存也跟着降到 $O(N)$。这正是 SRAM 有限但极快这一硬件特性倒逼出来的算法设计。


## 2. GPU 内部结构：SM、Tensor Core、Warp

HBM 和 SRAM 只是存储侧。计算侧，GPU 由若干个 SM（Streaming Multiprocessor）组成。每个 SM 内部包含若干 CUDA core、专门的 Tensor Core、寄存器文件和 shared memory。H100 SXM 有 132 个 SM，每 SM 配 256 KB shared memory 和 256 KB 寄存器。

调度单位是 warp：一个 warp 是 32 个线程，SM 一次发射并管理若干 warp。程序员写 CUDA 时通常不直接感知 warp，但写高性能 kernel 时需要按 32 对齐来组织数据，避免 warp 分支发散。


In [ ]:
# === H100 SXM 的计算资源 ===
sm_count = 132                  # SM 数量
shared_mem_per_sm_kb = 256      # 每 SM 的 shared memory
register_per_sm_kb = 256        # 每 SM 的寄存器文件

print(f"H100 SXM 关键参数：")
print(f"  SM 数量:            {sm_count}")
print(f"  每 SM shared memory: {shared_mem_per_sm_kb} KB")
print(f"  每 SM 寄存器:        {register_per_sm_kb} KB")
print(f"  总 shared memory:   {sm_count * shared_mem_per_sm_kb / 1024:.1f} MB ≈ SRAM 总量")
print()
print(f"warp 大小固定是 32 线程（NVIDIA 所有架构都一样）。")
print(f"程序员写 kernel 时按 32 对齐访问，能让 warp 内线程同时执行。")


### Tensor Core 与 FP8

Tensor Core 是 NVIDIA 从 Volta 架构（V100，2017）开始引入的专用矩阵乘单元。一个 Tensor Core 指令（MMA，Matrix Multiply-Accumulate）能在硬件层面完成一个小矩阵乘加：$D = A \times B + C$，其中 $A, B, C, D$ 是 $16\times16$ 或更小的 fragment。

不同架构的 Tensor Core 支持不同数据类型：

| 架构 | 代表 GPU | 关键数据类型 |
|:---|:---|:---|
| Volta | V100 | FP16 |
| Ampere | A100 | FP16、BF16、TF32、INT8 |
| Hopper | H100 | 上述 + FP8（E4M3、E5M2） |
| Blackwell | B200 | 上述 + FP4、FP6 |

Hopper 引入 FP8 Tensor Core 是 LLM 训练和推理的关键节点。FP8 用 8 bit 表示一个浮点数，相比 FP16 显存和带宽都减半，相比 BF16 还能保留接近的数值范围（E4M3 有 4 位指数 + 3 位尾数，E5M2 有 5 位指数 + 2 位尾数）。在合适的数据范围内，FP8 训练能跑出接近 BF16 的精度，速度却快得多。


In [ ]:
# === FP8 两种格式对比 ===
# E4M3: 1 符号 + 4 指数 + 3 尾数 → 精度高、范围小，适合前向
# E5M2: 1 符号 + 5 指数 + 2 尾数 → 精度低、范围大，适合反向梯度

print(f"{'格式':<8} {'符号位':>6} {'指数位':>6} {'尾数位':>6} {'典型用途':<14}")
print("-" * 50)
print(f"{'E4M3':<8} {'1':>6} {'4':>6} {'3':>6} {'前向权重':<14}")
print(f"{'E5M2':<8} {'1':>6} {'5':>6} {'2':>6} {'反向梯度':<14}")
print()
print("FP8 相比 BF16 的好处：")
print("  - 显存：相同张量只需一半字节（1 byte vs 2 byte）")
print("  - 带宽：HBM 读写量减半，对带宽受限的算子收益明显")
print("  - 算力：H100 FP8 Tensor Core 峰值算力是 BF16 的 2 倍")
print()
print("代价：数值范围窄，需要 per-tensor 或 per-row 缩放因子避免溢出。")


## 3. 多卡互联：PCIe、NVLink、NVSwitch、InfiniBand

单卡训练受限于显存和算力，规模稍大的模型必须多卡。多卡之间传数据（梯度、参数、KV cache）的带宽就成了瓶颈。NVIDIA 体系内有四种主流连接方式，带宽量级从低到高依次是 PCIe、NVLink、NVSwitch、InfiniBand（后两者定位不同）。


In [ ]:
# === 多卡互联带宽量级对比 ===
# 数值取自 NVIDIA 公开规格（典型值，单向带宽）

interconnects = [
    # (名称, 单向带宽 GB/s, 覆盖范围, 典型用途)
    ("PCIe 4.0 x16",       32,  "节点内",  "GPU-CPU、GPU-NVMe、低端 GPU-GPU"),
    ("PCIe 5.0 x16",       64,  "节点内",  "H100 PCIe 版本与 CPU 互联"),
    ("NVLink 3.0",        300,  "节点内",  "A100 之间直连"),
    ("NVLink 4.0",        450,  "节点内",  "H100 之间直连"),
    ("NVLink 5.0",        900,  "节点内",  "B200 之间直连"),
    ("InfiniBand HDR",     50,  "跨节点",  "200 Gbps 网络换算"),
    ("InfiniBand NDR",    100,  "跨节点",  "400 Gbps 网络换算"),
    ("InfiniBand XDR",    200,  "跨节点",  "800 Gbps 网络换算"),
]

print(f"{'名称':<20} {'带宽(GB/s)':>12} {'覆盖':>8}  典型用途")
print("-" * 80)
for name, bw, scope, use in interconnects:
    print(f"{name:<20} {bw:>12} {scope:>8}  {use}")

print()
print("关键观察：节点内 NVLink 是 PCIe 的 6-10 倍；跨节点 InfiniBand 比节点内 NVLink 慢 4-9 倍。")
print("这意味着「同一节点内的两张 GPU」和「跨节点的两张 GPU」通信速度差异巨大。")


### 为什么 NVLink 比 PCIe 快 6 到 10 倍

PCIe 是通用总线标准，原本为 CPU 和各种外设（网卡、显卡、硬盘）设计，协议本身有较大开销，每笔传输都要经过 PCIe 控制器、root complex 等多层。NVLink 是 NVIDIA 专为 GPU 之间点对点通信设计的协议，用差分信号线直接连两颗 GPU 的 die，绕开 PCIe 控制器，且支持 cache coherence（一致性）和直接内存访问（Peer-to-Peer，P2P）。

物理实现上，H100 SXM 每颗芯片有 4 个 NVLink 4.0 chip，每个 chip 提供 50 GB/s 单向带宽，合计 200 GB/s 单向、450 GB/s 双向（NVIDIA 习惯按双向算）。B200 把单 chip 带宽提到 100 GB/s，总 NVLink 带宽到 900 GB/s 双向。相比之下 PCIe 5.0 x16 单向只有 64 GB/s，物理带宽上限差一个数量级。


In [ ]:
# === NVLink vs PCIe 的带宽差距（以 H100 为例） ===
nvlink4_bidir = 450     # NVLink 4.0 双向带宽 GB/s
pcie5_unidir = 64       # PCIe 5.0 x16 单向带宽 GB/s

# 数据搬运演示：两张 GPU 之间传一个 70B 模型的 BF16 参数
P = 70e9
param_bytes = 2 * P     # BF16 每 param 2 bytes

t_nvlink = param_bytes / (nvlink4_bidir * 1e9)
t_pcie   = param_bytes / (pcie5_unidir * 1e9)

print(f"传输 70B 模型 BF16 参数（约 {param_bytes / 1e9:.0f} GB）：")
print(f"  走 NVLink 4.0: {t_nvlink:.2f} 秒")
print(f"  走 PCIe 5.0:  {t_pcie:.2f} 秒")
print(f"  速度差距: {t_pcie / t_nvlink:.1f} 倍")
print()
print("这就是为什么高带宽训练几乎都采用 NVLink，PCIe 只在低成本场景或推理侧使用。")


### NVSwitch：节点内全互联

NVLink 是点对点协议：两颗 GPU 之间一条 NVLink 通道。节点内 GPU 数量多了，单靠每对 GPU 拉 NVLink 会让拓扑非常复杂。H100 这一代 8 卡节点共有 28 对 GPU 互联（$C_8^2 = 28$），如果每对都拉满 NVLink 需要大量 chip。

NVSwitch 是 NVIDIA 的解决方案：在节点内放一个或多个 NVSwitch 芯片，所有 GPU 各自用 NVLink 接到 switch 上，switch 负责转发。这样任意两颗 GPU 之间都能跑满 NVLink 带宽，不需要为每对 GPU 单独拉线。DGX H100 上有 4 个第三代 NVSwitch，让 8 颗 H100 之间形成全互联（all-to-all）拓扑。

类比的话，NVLink 像两台机器之间直连一根网线，NVSwitch 像把它们都插到一台交换机上。


### InfiniBand：跨节点网络

节点内有 NVSwitch 解决全互联，跨节点怎么办？标准的以太网延迟太高、拥塞控制不可控，不适合大规模分布式训练。InfiniBand（IB）是专为高性能计算设计的网络协议，特点是低延迟（微秒级）、高带宽（HDR 200 Gbps、NDR 400 Gbps、XDR 800 Gbps）、硬件级可靠传输。

每个 GPU 节点通常配 4 到 8 张 IB 网卡（HCA，Host Channel Adapter），网卡通过 PCIe 5.0 连到 CPU，或通过 NVLink Switch 直连 GPU（这是 NVIDIA Magnum IO 架构的做法，称为「GPU Direct RDMA」）。GPU Direct RDMA 让一张 GPU 的显存直接通过 IB 网卡写到另一节点上某张 GPU 的显存，绕开 CPU 内存中转，延迟进一步降低。


In [ ]:
# === 跨节点传输：通过 IB NDR 传 70B 模型 ===
# NDR 单链路 400 Gbps = 50 GB/s

ib_ndr_gbps = 400          # Gbps
ib_ndr_gbs = ib_ndr_gbps / 8   # 转换为 GB/s

P = 70e9
param_bytes = 2 * P

# 通常每节点 8 张 IB 网卡，聚合带宽 8 × 50 = 400 GB/s
n_hca = 8
agg_bw = n_hca * ib_ndr_gbs

t_single = param_bytes / (ib_ndr_gbs * 1e9)
t_agg = param_bytes / (agg_bw * 1e9)

print(f"IB NDR 单链路: {ib_ndr_gbs:.0f} GB/s ({ib_ndr_gbps} Gbps)")
print(f"8 张 IB 聚合: {agg_bw:.0f} GB/s")
print()
print(f"跨节点传 70B BF16 参数 ({param_bytes / 1e9:.0f} GB)：")
print(f"  单链路: {t_single:.2f} 秒")
print(f"  8 链路聚合: {t_agg:.2f} 秒")
print()
print("相比节点内 NVLink 的 1.16 秒，跨节点慢约 2-3 倍（即便用满 8 张 IB）。")
print("这就是为什么 expert parallelism、tensor parallelism 的选择要仔细考虑拓扑。")


## 4. 典型拓扑：DGX H100

把上面几种连接方式拼起来，看一张 DGX H100 节点的完整拓扑。DGX H100 是 NVIDIA 官方的高密度训练节点，单节点有 8 颗 H100 SXM，是这个时代的「标准训练单元」。


### DGX H100 节点内拓扑

DGX H100 单节点的关键结构：

- 8 颗 H100 SXM GPU，每颗 80 GB HBM
- 4 颗第三代 NVSwitch 芯片，构成 8 卡全互联
- 每颗 H100 有 4 个 NVLink 4.0 chip，合计 900 GB/s 双向带宽（实际配置按 450 GB/s 双向算）
- 2 颗 Intel Sapphire Rapids CPU，通过 PCIe 5.0 连接 GPU 和外设
- 8 张 ConnectX-7 IB 网卡（400 Gbps NDR），每张对应一颗 GPU
- 2 张 BlueField-3 DPU，负责存储和网络协议卸载

这套设计让节点内任意两颗 GPU 之间都有 450 GB/s 双向带宽；跨节点通过 IB NDR 单链路 50 GB/s，每节点 8 链路聚合 400 GB/s。


In [ ]:
# === DGX H100 节点内 vs 跨节点带宽对比 ===
# 节点内：8 卡全互联
intra_link = 450          # 任意两卡之间 NVLink 双向 GB/s
n_pairs_intra = 8 * 7 // 2   # C(8,2) = 28 对
intra_agg = intra_link * n_pairs_intra

# 跨节点：每节点 8 张 IB NDR
ib_link = 50              # 单链路 GB/s
n_hca = 8
inter_agg = ib_link * n_hca

print(f"DGX H100 节点内全互联总带宽: {intra_agg} GB/s")
print(f"  (28 对 GPU × 450 GB/s = {intra_agg} GB/s)")
print()
print(f"DGX H100 跨节点聚合带宽: {inter_agg} GB/s")
print(f"  (8 张 IB × 50 GB/s = {inter_agg} GB/s)")
print()
print(f"节点内带宽是跨节点的 {intra_agg / inter_agg:.1f} 倍。")
print("这是「拓扑感知调度」的根本原因：把通信密集的操作放在节点内。")


### 集群网络层级

把多个 DGX 节点连起来构成训练集群，网络分三层：

- **节点内**：NVLink + NVSwitch，全互联，带宽最高
- **rack 内（机架内）**：通常一个 rack 放 2-4 个 DGX 节点，通过 IB switch 互联
- **跨 rack**：通过 spine switch（核心交换机）连接多个 rack，带宽进一步聚合

一个万卡规模的训练集群通常有约 1000-1500 个 DGX 节点，分布在几十个 rack 上。这种多层结构意味着任意两个 GPU 之间的通信路径可能完全不同——同节点内是 NVLink，同 rack 内是 1-2 跳 IB，跨 rack 是 3-4 跳 IB。延迟和带宽都逐层降低。


In [ ]:
# === 集群网络层级带宽估算 ===
layers = [
    # (层级, 单跳带宽 GB/s, 典型跳数)
    ("节点内 (NVLink)",   450, 1),
    ("rack 内 (IB NDR)",   50, 1),
    ("跨 rack (spine)",    50, 2),
]

print(f"{'层级':<24} {'单跳带宽':>12} {'典型总带宽':>12}")
print("-" * 55)
for name, bw, hops in layers:
    eff = bw / hops
    print(f"{name:<24} {bw:>10} GB/s {eff:>10.1f} GB/s")

print()
print("带宽逐层下降，所以调度器要尽量把通信密集的任务留在节点内。")


## 5. 拓扑感知调度

理解了带宽差距，就能解释为什么分布式训练调度器要「拓扑感知」（topology-aware）。一个朴素的想法是：随机把 N 个 GPU 分配给训练任务。但这样可能把两个需要频繁通信的 rank 放到跨节点位置，导致每步训练都在等跨节点 IB。

正确的做法是优先把任务放在「亲和度高的拓扑子集」里。例如：

- 数据并行（DDP/FSDP）：每步做 all-reduce 同步梯度，通信量大，但可以流水化。通常把 group 排成 ring 或 tree，让每跳都走 NVLink。优先放在同一节点内，跨节点时尽量同 rack。
- Tensor Parallelism：把单层矩阵乘切到多卡上，每步前向/反向都要 all-reduce。延迟敏感，**几乎必须放同一节点内**走 NVLink。
- Pipeline Parallelism：相邻 stage 之间传激活值，单向流。通信量比 TP 小，但延迟要低，通常放在相邻 GPU 上。
- Expert Parallelism（MoE）：每步 token 要路由到远端 expert，通信模式是 all-to-all，对带宽和拓扑都极敏感，下面单独展开。


In [ ]:
# === 不同并行策略对带宽的敏感度 ===
strategies = [
    # (策略, 每步通信模式, 通信量级, 拓扑要求)
    ("数据并行 (DDP/FSDP)",   "all-reduce 梯度",       "大",    "节点内优先，跨节点用 IB"),
    ("Tensor Parallelism",   "all-reduce 激活",        "中",    "强制节点内 (NVLink)"),
    ("Pipeline Parallelism", "P2P 激活值",             "小-中", "相邻 GPU 即可"),
    ("Expert Parallelism",   "all-to-all token 路由",  "大",    "节点内优先，跨节点需要 NVLink+IB 双满配"),
]

print(f"{'策略':<26} {'通信模式':<22} {'通信量':<8} 拓扑要求")
print("-" * 90)
for s, mode, vol, req in strategies:
    print(f"{s:<26} {mode:<22} {vol:<8} {req}")
print()
print("关键观察：TP 几乎不能跨节点；EP 通信模式最重，对拓扑要求最高。")


## 6. MoE 的 Expert Parallelism：为什么对网络拓扑极敏感

混合专家模型（Mixture of Experts，MoE）把 FFN 层替换成若干个并行的 expert 网络，每个 token 只激活其中少数几个（通常是 2 个，1 个共享 + 1 个路由）。Expert Parallelism（EP）是一种并行策略：把不同的 expert 分布到不同 GPU 上。例如 8 个 expert 放到 8 张卡，每张卡负责 1 个 expert 的前向反向。

EP 的通信模式是 **all-to-all**：

1. 每个 GPU 算出本地 token 应该路由到哪些 expert
2. 把目标 expert 在远端 GPU 上的 token 通过网络发过去
3. 远端 GPU 收到 token，做 expert 前向
4. 把结果再 all-to-all 发回原 GPU

每步训练要来回两次 all-to-all，通信的数据量等于 batch_size × seq_len × hidden_dim × 激活 expert 数。这个通信模式对带宽和拓扑都极敏感，因为每张 GPU 都可能要和任意其他 GPU 交换 token。

跨节点 EP 的代价尤其高。假设 8 个 expert 分布在两个节点上，每节点 4 个 expert。本地路由的 token 走 NVLink（450 GB/s），跨节点的 token 走 IB（50 GB/s 单链路）。如果 token 在 expert 之间均匀分布，约 3/8 的 token 要跨节点——这部分就成了瓶颈。这就是为什么大规模 MoE 训练（如 DeepSeek-MoE、Mixtral）几乎必须用满配 IB，且调度器要把同一 EP group 的 GPU 优先放在同一节点或同一 rack 内。


In [ ]:
# === EP all-to-all 通信量估算 ===
# 假设：8 卡节点，每卡 1 个 expert，batch 内 token 均匀路由到 8 个 expert
# 每个 token 是 hidden_dim 维向量

batch_size = 4096        # 全局 batch
seq_len = 4096
hidden_dim = 4096        # 模型 hidden size
expert_topk = 2          # 每 token 激活 2 个 expert（含共享）
bf16_bytes = 2

# 每 token 发送 topk 次（路由到 topk 个 expert）
# all-to-all 单向：每卡发出 batch*seq*topk/8 个 token，每个 token 是 hidden_dim 维
tokens_per_card = batch_size * seq_len * expert_topk / 8
bytes_per_token = hidden_dim * bf16_bytes

# 单卡单向发送字节数
send_bytes = tokens_per_card * bytes_per_token
# all-to-all 一个方向 + 一个方向回来 = 2 倍
round_trip = send_bytes * 2

print(f"EP 通信量估算（batch={batch_size}, seq={seq_len}, hidden={hidden_dim}, topk={expert_topk}）:")
print(f"  单卡发送 token 数: {tokens_per_card:.0f}")
print(f"  单 token 大小:     {bytes_per_token} bytes")
print(f"  单卡单向发送:      {send_bytes / 1e9:.2f} GB")
print(f"  往返通信量:        {round_trip / 1e9:.2f} GB")
print()
print(f"传输时间（NVLink 4.0, 450 GB/s）:  {round_trip / 450:.3f} 秒")
print(f"传输时间（IB NDR 单链路, 50 GB/s）: {round_trip / 50:.3f} 秒")
print()
print("跨节点传输时间约是节点内的 9 倍。")
print("MoE 训练中如果 EP 跨节点，all-to-all 往往是单步训练的主要时间开销。")


### EP 的部署策略

由于 EP 对带宽极度敏感，生产实践中常见的部署模式有几种：

- **节点内 EP**：把 expert 数量限制在节点内（8 卡 8 expert 或 16 expert）。所有 all-to-all 走 NVLink。优点是快，缺点是模型规模受限。
- **跨节点 EP + NVLink-first 调度**：跨节点 EP 必须用满 IB，调度器把同一 EP group 放在「网络最近」的 GPU 上。NCCL 会自动检测 `nvidia-smi topo -m` 输出，优先选择 NVLink 路径。
- **Expert Duplication**：把热门 expert 复制到多个 GPU 上，减少跨节点路由。代价是参数冗余。
- **DeepEP 等专用通信库**：DeepSeek 开源的 MoE 通信库，针对 EP 的 all-to-all 做了低延迟优化，支持节点内 NVLink + 跨节点 IB 的混合调度。


## 7. 实战：查看 GPU 拓扑的命令

实际机器上能通过几个 NVIDIA 工具直接看到上述硬件信息。下面列出最常用的几条命令，输出示例来自一台典型的 8 卡 H100 节点。


### nvidia-smi：基础信息

`nvidia-smi` 是 NVIDIA 驱动自带的命令，显示 GPU 的型号、显存、温度、利用率等。不加参数运行显示所有 GPU 概览：

```bash
nvidia-smi
```

要持续监控（类似 `top`），用 `nvtop` 或 `watch -n 1 nvidia-smi`。


### nvidia-smi topo -m：查看 GPU 间互联拓扑

这是排查多卡训练问题时最关键的命令。它显示一张 8×8 的矩阵，每行每列代表一张 GPU，单元格里的符号表示这两张 GPU 之间的连接方式：


In [ ]:
# === nvidia-smi topo -m 输出示例（8 卡 H100 节点） ===
# 实际机器上运行：nvidia-smi topo -m
# 下面是典型输出的字符串模拟

topo_output = '''
        GPU0    GPU1    GPU2    GPU3    GPU4    GPU5    GPU6    GPU7
GPU0     X      NV12    NV12    NV12    NV12    NV12    NV12    NV12
GPU1    NV12     X      NV12    NV12    NV12    NV12    NV12    NV12
GPU2    NV12    NV12     X      NV12    NV12    NV12    NV12    NV12
GPU3    NV12    NV12    NV12     X      NV12    NV12    NV12    NV12
GPU4    NV12    NV12    NV12    NV12     X      NV12    NV12    NV12
GPU5    NV12    NV12    NV12    NV12    NV12     X      NV12    NV12
GPU6    NV12    NV12    NV12    NV12    NV12    NV12     X      NV12
GPU7    NV12    NV12    NV12    NV12    NV12    NV12    NV12     X

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCI Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCI switches (PCIe bridge)
  PIX  = Connection traversing a single PCI switch (PCIe bridge)
  NV#  = Connection traversing a bonded set of # NVLinks
'''

print(topo_output)
print("符号解读：")
print("  NV12 表示两卡之间有 12 条 NVLink 通道（H100 的典型配置）")
print("  PIX/PXB 表示同 PCIe switch 内的连接，带宽较低")
print("  SYS 表示跨 NUMA 节点，通信最慢")
print()
print("如果矩阵里出现大量 PIX 或 SYS，说明 NVLink 没启用，多卡训练会慢很多。")


### nccl-tests：测量实际通信带宽

NVIDIA 的 NCCL（NVIDIA Collective Communications Library）提供了 `nccl-tests` 工具集，能实测 all-reduce、all-gather、all-to-all 等集合通信的实际带宽。下面这条命令测试 8 卡 all-reduce：

```bash
# 编译 nccl-tests 后运行 all_reduce_perf
./build/all_reduce_perf -b 8 -e 1G -f 2 -g 8
```

输出会显示不同消息大小下的 algbw（算法带宽）和 busbw（总线带宽），busbw 是衡量多卡互联效率的关键指标——一般 H100 节点的 all-reduce busbw 应该接近 NVLink 带宽的 80% 以上。


In [ ]:
# === 典型 all_reduce_perf 输出示例 ===
nccl_output = """
# size    count   type  redOp   time  algbw  busbw  error
        (B)    (elements)            (us)  (GB/s) (GB/s)
        8          2  float    sum   xx.x   x.xx   x.xx  0e+00
       16          4  float    sum   xx.x   x.xx   x.xx  0e+00
...
  8388608    2097152  float    sum   xx.x  xxx.x  xxx.x  0e+00
 16777216    4194304  float    sum   xx.x  xxx.x  xxx.x  0e+00
...
Out of bounds values : 0 OK
Avg bus bandwidth    : xxx.xx GB/s
"""

print(nccl_output)
print("健康指标：")
print("  - 8 卡 H100 NVLink 全互联的 all-reduce busbw 通常在 200-300 GB/s")
print("  - 跨节点 IB NDR 的 all-reduce busbw 通常在 30-50 GB/s")
print("  - 如果实测带宽远低于规格值，检查 NVLink 状态、NCCL 版本、P2P 是否启用")


## 8. 串起来：硬件特性如何影响算法选择

把本节内容回扣到前面的分布式训练讨论，几条关键对应关系：

- **Flash Attention 必需**是因为 SRAM 比 HBM 快近 10 倍，把 attention 分块留在 SRAM 是性能关键。
- **Tensor Parallelism 必须放节点内**是因为每步都 all-reduce，NVLink 比 IB 快 6-10 倍。
- **Pipeline Parallelism 可以跨节点**是因为相邻 stage 通信量小，IB 带宽够用。
- **FSDP/ZeRO 跨节点可行**是因为 all-reduce 流水化后能充分利用 IB 聚合带宽，但 busbw 是关键指标。
- **MoE EP 必须用满配 IB**是因为 all-to-all 通信模式最重，跨节点会成为主要瓶颈。
- **FP8 训练在 Hopper 上流行**是因为 FP8 Tensor Core 算力是 BF16 的 2 倍，且显存和带宽减半。


In [ ]:
# === 硬件特性 → 算法选择 的对应表 ===
mapping = [
    ("Flash Attention",         "SRAM 比 HBM 快 10 倍",       "把 attention 分块留在 SRAM"),
    ("Tensor Parallelism",       "NVLink 比 IB 快 6-10 倍",    "TP group 必须在节点内"),
    ("Pipeline Parallelism",     "P2P 通信量小",                "可跨节点，相邻 stage 即可"),
    ("FSDP / ZeRO-3",            "all-reduce 流水化",           "跨节点可行，busbw 是关键指标"),
    ("MoE Expert Parallelism",   "all-to-all 模式最重",         "必须满配 IB，节点内优先"),
    ("FP8 训练",                  "Hopper FP8 Tensor Core",     "算力 2× BF16，显存和带宽减半"),
]

print(f"{'算法 / 策略':<26} {'硬件依据':<28} 选择")
print("-" * 85)
for algo, hw, choice in mapping:
    print(f"{algo:<26} {hw:<28} {choice}")


## 小结

确认已经搞懂这些：

- [ ] HBM 容量大但相对慢，SRAM 容量小但极快（H100 上 SRAM 估算带宽约是 HBM 的 10 倍）
- [ ] Flash Attention 把 attention 分块留在 SRAM，HBM 读写从 $O(N^2)$ 降到 $O(N)$
- [ ] GPU 由若干 SM 组成，每 SM 内含 Tensor Core；warp 是 32 线程的调度单位
- [ ] Hopper 引入 FP8（E4M3、E5M2）Tensor Core，相比 BF16 算力翻倍、显存带宽减半
- [ ] PCIe 5.0 单向 64 GB/s，NVLink 4.0 双向 450 GB/s，差距约 6-10 倍
- [ ] NVSwitch 让节点内 8 卡全互联，跨节点用 InfiniBand（NDR 400 Gbps）
- [ ] DGX H100 节点内全互联带宽约 28 × 450 GB/s，跨节点聚合约 8 × 50 GB/s
- [ ] Tensor Parallelism 几乎必须放节点内；EP all-to-all 对拓扑最敏感
- [ ] `nvidia-smi topo -m` 是排查多卡拓扑问题的首选命令


## 作业

> 可以让 AI 帮忙解释思路、检查方向，但不建议直接让 AI 「做完这道题」。

**作业 1：估算 Flash Attention 的 HBM 读写节省**

序列长度 $N = 8192$，hidden_dim $d = 4096$，head 数 $h = 32$。朴素 attention 的中间矩阵 $QK^T$ 形状是 $[h, N, N]$，BF16 存储。计算这个中间矩阵的字节数，并说明 Flash Attention 能把它降到多少。

小提示：朴素实现要把 $[h, N, N]$ 写到 HBM 再读回来；Flash Attention 用 online softmax 分块计算，只保留 $[h, N]$ 的归一化因子和累加结果。


In [ ]:
# 作业 1：Flash Attention 的 HBM 读写节省
N = 8192
d = 4096
h = 32
bf16_bytes = 2

# TODO: 计算朴素 attention 中间矩阵的字节数（单位 GB）
naive_bytes_gb = None

# TODO: Flash Attention 保留的中间结果字节数（单位 GB）
# 提示：online softmax 只保留 [h, N] 的 running max 和 running sum
flash_bytes_gb = None

assert naive_bytes_gb is not None, "请先计算朴素 attention 中间矩阵大小"
assert flash_bytes_gb is not None, "请先计算 Flash Attention 中间结果大小"

expected_naive = h * N * N * bf16_bytes / 1e9
expected_flash = h * N * 2 * 2 * bf16_bytes / 1e9   # running max + running sum，各 h*N 个元素

assert abs(naive_bytes_gb - expected_naive) < 0.1, f"朴素中间矩阵应为 {expected_naive:.2f} GB"
assert abs(flash_bytes_gb - expected_flash) < 0.001, f"Flash 中间结果应为 {expected_flash:.4f} GB"

print(f"✅ 作业 1 通过：")
print(f"   朴素 attention 中间矩阵: {naive_bytes_gb:.2f} GB")
print(f"   Flash Attention 中间结果: {flash_bytes_gb:.4f} GB")
print(f"   节省比例: {1 - flash_bytes_gb / naive_bytes_gb * 100:.2f}%")


**作业 2：跨节点 vs 节点内传输时间**

70B 模型的 BF16 参数，要在两张 GPU 之间传输。如果两张卡同节点（走 NVLink 4.0），传输时间是多少秒？如果跨节点（走 IB NDR 单链路），传输时间又是多少秒？速度差几倍？

小提示：NVLink 4.0 双向带宽 450 GB/s，IB NDR 单链路 50 GB/s。参数总量是 70e9，BF16 每 param 2 bytes。


In [ ]:
# 作业 2：跨节点 vs 节点内传输时间
P = 70e9
bf16_bytes = 2
param_bytes = P * bf16_bytes

nvlink_bw = 450       # NVLink 4.0 双向 GB/s
ib_ndr_bw = 50        # IB NDR 单链路 GB/s

# TODO: 计算两种传输时间（秒）
t_intra = None       # 节点内 NVLink
t_inter = None       # 跨节点 IB

assert t_intra is not None and t_inter is not None, "请先计算两种传输时间"

expected_intra = param_bytes / (nvlink_bw * 1e9)
expected_inter = param_bytes / (ib_ndr_bw * 1e9)
assert abs(t_intra - expected_intra) < 0.01, f"节点内时间应为 {expected_intra:.2f} 秒"
assert abs(t_inter - expected_inter) < 0.01, f"跨节点时间应为 {expected_inter:.2f} 秒"

print(f"✅ 作业 2 通过：")
print(f"   参数总量: {param_bytes / 1e9:.0f} GB")
print(f"   节点内 (NVLink 4.0): {t_intra:.2f} 秒")
print(f"   跨节点 (IB NDR):    {t_inter:.2f} 秒")
print(f"   速度差距: {t_inter / t_intra:.1f} 倍")


**作业 3：识别拓扑问题**

假设在 8 卡节点上跑 `nvidia-smi topo -m`，输出矩阵里全是 `PIX` 而不是 `NV12`。这时运行 DDP 训练会观察到什么现象？应该优先检查什么？

小提示：`PIX` 表示两张 GPU 通过同一个 PCIe switch 相连，没有走 NVLink。NVLink 不工作通常和驱动、CUDA 版本、电源模式、PCIe 拓扑配置有关。


In [ ]:
# 作业 3：识别拓扑问题（概念题，填空）
# 题目：topo -m 显示全是 PIX 时，DDP 训练会怎样？

# 填空：把正确的字符串填到下面两个变量里
# 现象（从 options_phenomenon 选一个）：
#   "训练速度正常，与 NVLink 一致"
#   "训练速度大幅下降，all-reduce 成为瓶颈"
#   "训练无法启动，NCCL 报错"
phenomenon = None

# 优先检查项（从 options_check 选一个）：
#   "nvidia 驱动是否支持 NVLink、CUDA 版本是否匹配、BIOS 中 NVLink bridge 是否启用"
#   "增加 batch size"
#   "更换 PyTorch 版本"
check_action = None

options_phenomenon = {
    "训练速度大幅下降，all-reduce 成为瓶颈"
}
options_check = {
    "nvidia 驱动是否支持 NVLink、CUDA 版本是否匹配、BIOS 中 NVLink bridge 是否启用"
}

assert phenomenon in options_phenomenon, "现象判断有误：PIX 意味着 NVLink 没启用，all-reduce 只能走慢得多的 PCIe"
assert check_action in options_check, "排查方向不对：先确认 NVLink 硬件和驱动层面是否正常"

print("✅ 作业 3 通过：")
print(f"   现象: {phenomenon}")
print(f"   排查: {check_action}")


## 参考资料

- NVIDIA, [H100 Tensor Core GPU Architecture Whitepaper](https://resources.nvidia.com/en-us-tensor-core), 2022
- NVIDIA, [Hopper FP8 Tensor Cores](https://www.nvidia.com/en-us/data-center/hopper-architecture/), 2022
- NVIDIA, [NVLink and NVSwitch](https://www.nvidia.com/en-us/data-center/nvlink/), 2023
- NVIDIA, [DGX H100 System Architecture](https://www.nvidia.com/en-us/data-center/dgx-h100/), 2023
- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), 2022
- NVIDIA, [NCCL Tests Documentation](https://github.com/NVIDIA/nccl-tests), 2023
- InfiniBand Trade Association, [InfiniBand Architecture Specification](https://www.infinibandta.org/), 2024
